In [ ]:
import numpy as np
import time
import cv2
import os
import zlib
from sdlarch_rl import make
import pygame
from IPython.display import Audio
from stable_baselines3 import PPO
from stable_baselines3.common.atari_wrappers import WarpFrame, MaxAndSkipEnv
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.env_util import make_vec_env
from sdlarch_rl.utils.utils import get_latest_model, TrainAndLoggingCallback, FrameSkip, TimeLimit, RealExcludeButtonsWrapper
from sdlarch_rl.utils.discretizer import MainDiscretizer
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.callbacks import CallbackList, EvalCallback
from pathlib import Path

import logging
# import multiprocessing as mp
# mp.set_start_method("spawn", force=True)
logging.basicConfig(level=logging.DEBUG)

NUM_ENV = 1
SAVE_DIR="./model-gt3"

MAX_STEPS= 12_000

SAVE_DIR = Path(SAVE_DIR)
combos = [
   [], # noop

    # # only turn without acelerate
    # ["LEFT"],
    # ["RIGHT"],

    # acelerate
    ['B'],
    ["LEFT", 'B'],
    ["RIGHT", 'B'],

    #  brake
    ["Y"],
    
    # reverse
    ["X"],
]

def make_env(env_id):
    def _init():
        env = make(
            "GranTurismo3-Ps2", 
            statename="middle_field",
            # render_mode="human"
        )
        
        buttons = env.unwrapped.buttons
        to_exclude = ["UP", "DOWN", "START", "SELECT", "R1", "L1", "L2", "R2", "L3", "R3", "A"]
        
        env = RealExcludeButtonsWrapper(env, buttons, to_exclude)
        
        env = WarpFrame(env, width=96, height=96)
        env = FrameSkip(env, skip=4)
        env = TimeLimit(env, max_steps=MAX_STEPS)

        return env
    return _init

    
# env = make_vec_env(make_env(), n_envs=NUM_ENV)
env = make_vec_env(make_env(15), n_envs=NUM_ENV, vec_env_cls=DummyVecEnv)
env = VecFrameStack(env, 4, channels_order='last')

latest_model_path = get_latest_model(SAVE_DIR)

print("loading from: " + str(latest_model_path))

model = PPO.load(
# model = RecurrentPPO.load(
    str(latest_model_path), 
    env=env, 
    verbose=0, 
)

obs = env.reset()

frame_count = 0
start_time = time.time()


while True:
    action, _ = model.predict(obs, deterministic=False)

    env.render() 

    # if action[0] == 7:
    #     action[0] = 5
    
    obs, reward, done, info = env.step(action)

    frame_count += 1

    elapsed = time.time() - start_time

    if elapsed >= 1.0:
        fps = frame_count / elapsed
        print(f"FPS: {fps:.2f}")

        # reset
        frame_count = 0
        start_time = time.time()


env.close()


D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


loading from: model-gt3\best_model_195000


D:\Python311\Lib\site-packages\stable_baselines3\common\save_util.py:449: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  th_object = th.load(file_content, map_location=device

FPS: 0.52
FPS: 25.84
FPS: 28.53
FPS: 26.69
FPS: 25.11
FPS: 26.92
FPS: 29.71
FPS: 30.98
FPS: 30.53
FPS: 23.87
FPS: 28.83
FPS: 28.99
FPS: 28.45
FPS: 29.75
FPS: 29.25
FPS: 27.24
FPS: 26.75
FPS: 28.22
FPS: 28.79
FPS: 26.95
FPS: 26.37
FPS: 26.23
